# CEREBRO PoC — Ground Truth & Benchmark Framework

**Benchmark:** CEREBRO-GT-v0.1  
**Experiment:** EXP-GT-001

## Objective

Establish a controlled multimodal ground-truth dataset for evaluating
CEREBRO's ingestion, transformation, knowledge extraction, retrieval,
and recollection pipelines.

The benchmark separates:

- source evidence
- ownership
- authorship
- ground truth
- derived knowledge
- provenance

## CEREBRO Integrity Rules

**CIF-01 — No derived knowledge without provenance.**

Every derived representation must identify the artifact and processing
lineage from which it originated.

**CIF-02 — Source evidence must remain resolvable.**

Every trusted knowledge fragment must resolve back to its original
source artifact and, where possible, its precise source location such
as page, slide, timestamp, section, or image region.

## Canonical Artifact Model

```mermaid
flowchart TD
    U[USER]
    A[ARTIFACT]
    P[PERSON / ORGANIZATION]
    K[KNOWLEDGE FRAGMENT]
    S[SOURCE LOCATION]

    U -->|owns / ingests| A
    P -->|authors| A
    A -->|contains| K
    K -->|references| S
```

### Define canonical artifact model

In [1]:
artifact = {
    "artifact_id": "ART-0001",
    "owner_id": "USER-001",

    "title": "CEREBRO Benchmark Text",
    "modality": "text",
    "mime_type": "text/plain",
    "language": "en",

    "authors": [
        {
            "author_id": "PERSON-001",
            "name": "Benchmark Author"
        }
    ],

    "source": {
        "filename": "benchmark_001.txt",
        "storage_uri": "poc/data/raw/text/benchmark_001.txt",
        "sha256": None
    },

    "benchmark": {
        "dataset": "CEREBRO-GT-v0.1",
        "manually_verified": True
    }
}

artifact

{'artifact_id': 'ART-0001',
 'owner_id': 'USER-001',
 'title': 'CEREBRO Benchmark Text',
 'modality': 'text',
 'mime_type': 'text/plain',
 'language': 'en',
 'authors': [{'author_id': 'PERSON-001', 'name': 'Benchmark Author'}],
 'source': {'filename': 'benchmark_001.txt',
  'storage_uri': 'poc/data/raw/text/benchmark_001.txt',
  'sha256': None},
 'benchmark': {'dataset': 'CEREBRO-GT-v0.1', 'manually_verified': True}}

### Define the knowledge-fragment contract

In [2]:
knowledge_fragment = {
    "fragment_id": "KF-0001",

    "artifact_id": "ART-0001",
    "owner_id": "USER-001",

    "content": "Example ground-truth knowledge fragment.",

    "source_location": {
        "type": "text_span",
        "start_char": 0,
        "end_char": 40
    },

    "provenance": {
        "extraction_method": "manual_ground_truth",
        "parent_artifact_id": "ART-0001"
    },

    "manually_verified": True
}

knowledge_fragment

{'fragment_id': 'KF-0001',
 'artifact_id': 'ART-0001',
 'owner_id': 'USER-001',
 'content': 'Example ground-truth knowledge fragment.',
 'source_location': {'type': 'text_span', 'start_char': 0, 'end_char': 40},
 'provenance': {'extraction_method': 'manual_ground_truth',
  'parent_artifact_id': 'ART-0001'},
 'manually_verified': True}

### Integrity Test

Expectation: Will result to be successful and zero missing knowledge fragments.

In [3]:
def validate_fragment_provenance(fragment):
    required = [
        "fragment_id",
        "artifact_id",
        "owner_id",
        "source_location",
        "provenance"
    ]

    missing = [
        field for field in required
        if not fragment.get(field)
    ]

    return {
        "valid": len(missing) == 0,
        "missing": missing
    }


validate_fragment_provenance(knowledge_fragment)

{'valid': True, 'missing': []}

### Delberately break the provenance to test validation

Expectation: Will highlight the missing fragment.

In [ ]:
invalid_fragment = knowledge_fragment.copy()
invalid_fragment["artifact_id"] = None

validate_fragment_provenance(invalid_fragment)

{'valid': False, 'missing': ['artifact_id']}

### Cerebro Principle Enforcement process

```mermaid
flowchart TD
    K[Knowledge]
    Q1{Where did this come from?}
    A[ART-0001]
    Q2{Where is the evidence?}
    S[Actual Source Artifact]

    K --> Q1
    Q1 --> A
    A --> Q2
    Q2 --> S
```

### 1. Activate the environment